In [0]:
from pyspark.sql.functions import col, lower, trim, when, current_timestamp, coalesce, lit, expr

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

print("✓ Silver schema ready")

In [0]:
# First clean invoices from source
invoices_clean = (
    spark.table("azure_blob_storage.src_invoices")
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("invoice_status", coalesce(lower(trim(col("invoice_status"))), lit("none")))
    .withColumn("currency", coalesce(lower(trim(col("currency"))), lit("none")))
    .withColumn("region", coalesce(lower(trim(col("region"))), lit("none")))
    .withColumn("invoice_date", 
        expr("""
            coalesce(
                try_to_date(invoice_date, 'yyyy/MM/dd'),
                try_to_date(invoice_date, 'yyyy/M/d'),
                try_to_date(invoice_date, 'yyyy-MM-dd'),
                try_to_date(invoice_date, 'MM/dd/yyyy')
            )
        """)
    )
    .withColumn("ingestion_ts", current_timestamp())
    .dropDuplicates(["invoice_id"])
)

# Join with customers and regions
invoices = (
    invoices_clean.alias("inv")
    .join(
        spark.table("silver.customers").alias("cust"),
        col("inv.customer_id") == col("cust.customer_id"),
        "left"
    )
    .join(
        spark.table("silver.regions").alias("reg"),
        col("inv.region") == col("reg.region_name"),
        "left"
    )
    .select(
        col("inv.invoice_id"),
        col("inv.customer_id"),
        coalesce(col("cust.customer_name"), lit("none")).alias("customer_name"),
        coalesce(col("cust.customer_type"), lit("none")).alias("customer_type"),
        coalesce(col("cust.country"), lit("NONE")).alias("customer_country"),
        col("inv.invoice_date"),
        col("inv.invoice_month"),
        col("inv.currency"),
        col("inv.region"),
        coalesce(col("reg.region_code"), lit("none")).alias("region_code"),
        coalesce(col("reg.country"), lit("NONE")).alias("region_country"),
        col("inv.invoice_status"),
        col("inv.ingestion_ts")
    )
)

invoices.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.invoices")
print(f" invoices: {invoices.count()} records")

In [0]:
print("\nSample data (with customer and region info):")
display(spark.table("silver.invoices").limit(5))